In [1]:
# Cell 1 — Setup (same logic as your KAN-AD notebook)

!pip install --upgrade pip
!pip install toml torch torchinfo optuna
!pip install git+https://github.com/CSTCloudOps/EasyTSAD.git

# Clone repos
!rm -rf KAN-AD
!rm -rf datasets

!git clone https://github.com/CSTCloudOps/KAN-AD.git
!git clone https://github.com/CSTCloudOps/datasets.git
!mv datasets KAN-AD/datasets

%cd KAN-AD

# Path setup
import sys, os

project_path = os.getcwd()
if project_path not in sys.path:
    sys.path.append(project_path)

print("🟢 Paths configured.")
print("Current working directory:", project_path)

# Fix EasyTSAD import bug
!sed -i 's/TSData、/TSData/g; s/TSData,*/TSData/g' \
    /usr/local/lib/python3.12/dist-packages/EasyTSAD/DataFactory/__init__.py

# Test imports
from EasyTSAD.Controller import TSADController
from EasyTSAD.DataFactory import TSData
from EasyTSAD.Methods import BaseMethod

print("✅ EasyTSAD ready")

  Cloning https://github.com/CSTCloudOps/EasyTSAD.git to /tmp/pip-req-build-9kmgobm6
  Running command git clone --filter=blob:none --quiet https://github.com/CSTCloudOps/EasyTSAD.git /tmp/pip-req-build-9kmgobm6
  Resolved https://github.com/CSTCloudOps/EasyTSAD.git to commit 507e5533a142eec7eece4372c55e83bb3faff8a1
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Cloning into 'KAN-AD'...
remote: Enumerating objects: 18, done.
remote: Counting objects: 100% (18/18), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 18 (delta 1), reused 15 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (18/18), 61.77 KiB | 585.00 KiB/s, done.
Resolving deltas: 100% (1/1), done.
Cloning into 'datasets'...
remote: Enumerating objects: 4503, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 4503 (delta 2), reused 0 (delta 0), pack-r

In [2]:
import torch
import torch.nn as nn

class LSTMAutoencoder(nn.Module):
    def __init__(self, num_features, hidden_size=64, num_layers=1):
        super().__init__()

        self.num_features = num_features
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # Encoder
        self.encoder = nn.LSTM(
            input_size=num_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )

        # Decoder
        self.decoder = nn.LSTM(
            input_size=hidden_size,
            hidden_size=num_features,
            num_layers=num_layers,
            batch_first=True
        )

    def forward(self, x):
        """
        x: (B, W, F)
        """

        # Encode
        _, (hidden, _) = self.encoder(x)

        # Repeat latent vector across sequence length
        latent = hidden[-1].unsqueeze(1).repeat(1, x.size(1), 1)

        # Decode
        out, _ = self.decoder(latent)

        return out  # (B, W, F)


print("✅ LSTM Autoencoder ready")

✅ LSTM Autoencoder ready


In [3]:
# Cell 4 — LSTM Autoencoder as EasyTSAD Method

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import tqdm

from torch.utils.data import DataLoader
from EasyTSAD.Methods import BaseMethod


class LSTM_AE_TSAD(BaseMethod):

    def __init__(self, params: dict) -> None:
        super().__init__()

        self.__anomaly_score = None

        self.window = int(params["window"])
        self.batch_size = int(params["batch_size"])
        self.epochs = int(params["epochs"])
        self.lr = float(params["lr"])
        self.patience = int(params.get("patience", 5))

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        num_features = int(params["num_features"])
        hidden_size = int(params.get("hidden_size", 64))

        # ✅ LSTM Autoencoder backbone
        self.model = LSTMAutoencoder(
            num_features=num_features,
            hidden_size=hidden_size
        ).to(self.device)

        self.optimizer = optim.Adam(self.model.parameters(), lr=self.lr)
        self.scheduler = optim.lr_scheduler.StepLR(self.optimizer, step_size=5, gamma=0.75)

        self.mse_loss = nn.MSELoss()

        self.best_state = None

    # -----------------------------
    # Forward batch
    # -----------------------------
    def _forward_batch(self, x):
        """
        x: (B, W, F)
        """

        recon = self.model(x)

        loss = self.mse_loss(recon, x)

        # anomaly score = reconstruction error per window
        err = torch.mean((recon - x) ** 2, dim=(1, 2))  # (B,)

        return err, loss

    # -----------------------------
    # Train + Valid
    # -----------------------------
    def train_valid_phase(self, tsTrain):

        train_loader = DataLoader(
            MTSWindowDataset(tsTrain, "train", self.window),
            batch_size=self.batch_size,
            shuffle=True
        )

        valid_loader = DataLoader(
            MTSWindowDataset(tsTrain, "valid", self.window),
            batch_size=self.batch_size,
            shuffle=False
        )

        best_valid = float("inf")
        patience_counter = 0

        for epoch in range(1, self.epochs + 1):

            self.model.train()
            train_losses = []

            for x, _ in tqdm.tqdm(train_loader, desc=f"Train {epoch}"):

                x = x.to(self.device)

                self.optimizer.zero_grad(set_to_none=True)

                _, loss = self._forward_batch(x)

                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=5.0)

                self.optimizer.step()
                train_losses.append(float(loss.item()))

            # VALID
            self.model.eval()
            valid_losses = []

            with torch.no_grad():
                for x, _ in tqdm.tqdm(valid_loader, desc=f"Valid {epoch}"):

                    x = x.to(self.device)

                    _, loss = self._forward_batch(x)
                    valid_losses.append(float(loss.item()))

            train_loss = np.mean(train_losses)
            valid_loss = np.mean(valid_losses)

            print(f"Epoch {epoch} | train={train_loss:.6f} | valid={valid_loss:.6f}")

            self.scheduler.step()

            # Early stopping
            if valid_loss < best_valid:
                best_valid = valid_loss
                patience_counter = 0

                self.best_state = {
                    "model": {k: v.detach().cpu().clone() for k, v in self.model.state_dict().items()}
                }
            else:
                patience_counter += 1
                if patience_counter >= self.patience:
                    print("Early stopping")
                    break

        # restore best model
        assert self.best_state is not None
        self.model.load_state_dict(self.best_state["model"])

    # -----------------------------
    # Test phase
    # -----------------------------
    def test_phase(self, tsData):

        test_loader = DataLoader(
            MTSWindowDataset(tsData, "test", self.window),
            batch_size=self.batch_size,
            shuffle=False
        )

        self.model.eval()

        scores = []

        with torch.no_grad():
            for x, _ in test_loader:

                x = x.to(self.device)

                err, _ = self._forward_batch(x)
                scores.append(err.cpu().numpy())

        scores = np.concatenate(scores) if scores else np.array([])

        # pad to match original length
        if len(scores) == 0:
            padded = np.zeros(len(tsData.test))
        else:
            prefix = np.full(self.window, scores[0])
            padded = np.concatenate([prefix, scores])
            padded = padded[:len(tsData.test)]

        self.__anomaly_score = padded.astype(np.float64)

    # -----------------------------
    def anomaly_score(self) -> np.ndarray:
        return self.__anomaly_score


print("✅ LSTM Autoencoder EasyTSAD method ready")

✅ LSTM Autoencoder EasyTSAD method ready


In [4]:
# Cell 5 — Create config file for LSTM Autoencoder on SMAP

import os
import numpy as np

DATASET = "SMAP"
data_dir = f"/content/KAN-AD/datasets/MTS/{DATASET}/AllInOne"
train_arr = np.load(os.path.join(data_dir, "train.npy"))
NUM_FEATURES = train_arr.shape[1]

config_text = f"""\
[Data_Params]
preprocess = "z-score"
diff_order = 0

[Model_Params.Default]
window = 96
batch_size = 128
epochs = 50
lr = 0.001
patience = 5

# LSTM-specific
hidden_size = 64
num_features = {NUM_FEATURES}
"""

CFG_PATH_LSTM = os.path.join("kanad", "config_lstm_ae_smap.toml")

with open(CFG_PATH_LSTM, "w") as f:
    f.write(config_text)

print("✅ LSTM AE config written to:", CFG_PATH_LSTM)
print("SMAP features:", NUM_FEATURES)
print(open(CFG_PATH_LSTM).read())


✅ LSTM AE config written to: kanad/config_lstm_ae_smap.toml
SMAP features: 25
[Data_Params]
preprocess = "z-score"
diff_order = 0

[Model_Params.Default]
window = 96
batch_size = 128
epochs = 50
lr = 0.001
patience = 5

# LSTM-specific
hidden_size = 64
num_features = 25



In [5]:
# Cell 4.5 — MTSWindowDataset for LSTM Autoencoder

from torch.utils.data import Dataset
import numpy as np
import torch

class MTSWindowDataset(Dataset):
    def __init__(self, tsData, phase, window_size):
        self.window_size = int(window_size)

        if phase == "train":
            self.data = np.asarray(tsData.train)
        elif phase == "valid":
            self.data = np.asarray(tsData.valid)
        elif phase == "test":
            self.data = np.asarray(tsData.test)
        else:
            raise ValueError("phase must be train / valid / test")

        assert self.data.ndim == 2, f"Expected 2D MTS array, got {self.data.shape}"

        self.N, self.F = self.data.shape
        self.sample_num = max(self.N - self.window_size, 0)

    def __len__(self):
        return self.sample_num

    def __getitem__(self, idx):
        # window
        x = self.data[idx:idx + self.window_size]

        # for LSTM AE → target = input (reconstruction)
        return (
            torch.tensor(x, dtype=torch.float32),
            torch.tensor(x, dtype=torch.float32),
        )

print("✅ MTSWindowDataset ready for LSTM Autoencoder")

✅ MTSWindowDataset ready for LSTM Autoencoder


In [6]:
# Cell 6 — Train LSTM Autoencoder on ORIGINAL SMAP

from EasyTSAD.Controller import TSADController

gctrl = TSADController()

# -----------------------------
# Dataset (ORIGINAL SMAP)
# -----------------------------
gctrl.set_dataset(
    dataset_type="MTS",
    dirname="datasets",
    datasets=["SMAP"],   # original dataset (no holdout)
)

# -----------------------------
# Method
# -----------------------------
METHOD_NAME = "LSTM_AE_TSAD"
TRAINING_SCHEMA = "naive"

print("🚀 Training LSTM Autoencoder baseline on SMAP...")

gctrl.run_exps(
    method=METHOD_NAME,
    training_schema=TRAINING_SCHEMA,
    cfg_path=CFG_PATH_LSTM,
)

print("✅ Training finished")

(2026-06-05 22:52:49,063) [INFO]: 
                         
███████╗ █████╗ ███████╗██╗   ██╗    ████████╗███████╗ █████╗ ██████╗ 
██╔════╝██╔══██╗██╔════╝╚██╗ ██╔╝    ╚══██╔══╝██╔════╝██╔══██╗██╔══██╗
█████╗  ███████║███████╗ ╚████╔╝        ██║   ███████╗███████║██║  ██║
██╔══╝  ██╔══██║╚════██║  ╚██╔╝         ██║   ╚════██║██╔══██║██║  ██║
███████╗██║  ██║███████║   ██║          ██║   ███████║██║  ██║██████╔╝
╚══════╝╚═╝  ╚═╝╚══════╝   ╚═╝          ╚═╝   ╚══════╝╚═╝  ╚═╝╚═════╝ 
                                                                      
                         
INFO:logger:
                         
███████╗ █████╗ ███████╗██╗   ██╗    ████████╗███████╗ █████╗ ██████╗ 
██╔════╝██╔══██╗██╔════╝╚██╗ ██╔╝    ╚══██╔══╝██╔════╝██╔══██╗██╔══██╗
█████╗  ███████║███████╗ ╚████╔╝        ██║   ███████╗███████║██║  ██║
██╔══╝  ██╔══██║╚════██║  ╚██╔╝         ██║   ╚════██║██╔══██║██║  ██║
███████╗██║  ██║███████║   ██║          ██║   ███████║██║  ██║██████╔╝
╚══════╝╚═╝  ╚═╝╚═════

🚀 Training LSTM Autoencoder baseline on SMAP...


(2026-06-05 22:52:49,768) [INFO]:     [LSTM_AE_TSAD] handling dataset SMAP | curve AllInOne 
INFO:logger:    [LSTM_AE_TSAD] handling dataset SMAP | curve AllInOne 
Valid 1: 100%|██████████| 1056/1056 [00:04<00:00, 253.74it/s]


Epoch 1 | train=11470.694035 | valid=11469.648129


Valid 2: 100%|██████████| 1056/1056 [00:04<00:00, 235.97it/s]


...

Epoch 29 | train=11469.305109 | valid=11469.028825


Valid 30: 100%|██████████| 1056/1056 [00:04<00:00, 236.53it/s]


Epoch 30 | train=11469.531360 | valid=11469.035607


Valid 31: 100%|██████████| 1056/1056 [00:04<00:00, 225.60it/s]


Epoch 31 | train=11469.930836 | valid=11469.028052
Early stopping
✅ Training finished


In [7]:
# Cell 7 — Evaluate LSTM Autoencoder (same protocol as KAN)

from EasyTSAD.Evaluations.Protocols import (
    EventF1PA,
    PointF1PA,
    PointKthF1PA,
    PointAuprcPA,
)

method = "LSTM_AE_TSAD"
training_schema = "naive"

gctrl.set_evals([
    PointF1PA(),                  # main F1 (paper metric)
    EventF1PA(mode="squeeze"),
    PointKthF1PA(k=5),
    PointAuprcPA(),
])

gctrl.do_evals(method=method, training_schema=training_schema)

print("✅ Evaluation completed")

(2026-06-05 22:58:53,008) [INFO]: Register evaluations
INFO:logger:Register evaluations
(2026-06-05 22:58:53,010) [INFO]: Perform evaluations. Method[LSTM_AE_TSAD], Schema[naive].
INFO:logger:Perform evaluations. Method[LSTM_AE_TSAD], Schema[naive].
(2026-06-05 22:58:53,013) [INFO]:     [Load Data (All)] DataSets: SMAP 
INFO:logger:    [Load Data (All)] DataSets: SMAP 
(2026-06-05 22:58:53,058) [INFO]:     [LSTM_AE_TSAD] Eval dataset SMAP <<<
INFO:logger:    [LSTM_AE_TSAD] Eval dataset SMAP <<<
(2026-06-05 22:58:53,060) [INFO]:         [SMAP] Using default margins (0, 5)
INFO:logger:        [SMAP] Using default margins (0, 5)


✅ Evaluation completed


In [8]:
# Cell 8 — Load LSTM AE results and convert to comparison row

import json
import os

# -----------------------------
# Paths
# -----------------------------
avg_path = "/content/KAN-AD/Results/Evals/LSTM_AE_TSAD/naive/SMAP/avg.json"
all_path = "/content/KAN-AD/Results/Evals/LSTM_AE_TSAD/naive/SMAP/all.json"

assert os.path.exists(avg_path), f"avg.json not found: {avg_path}"
assert os.path.exists(all_path), f"all.json not found: {all_path}"

# -----------------------------
# Load
# -----------------------------
with open(avg_path, "r") as f:
    avg = json.load(f)

with open(all_path, "r") as f:
    all_scores = json.load(f)

# -----------------------------
# Print
# -----------------------------
print("=== 📊 AVERAGE RESULTS (LSTM AE - SMAP) ===")
for k, v in avg.items():
    print(f"{k}: {v}")

print("\n=== 📊 PER-SERIES RESULTS ===")
print("Number of series:", len(all_scores))
print("Example entry:")
print(list(all_scores.items())[0])

# -----------------------------
# Extract key metrics
# -----------------------------
row = {
    "Model": "LSTM Autoencoder",
    "Family": "Reconstruction",
    "Attention": "No",
    "Dataset": "SMAP",

    # Main metrics
    "F1": avg["best f1 under pa"]["f1"],
    "Precision": avg["best f1 under pa"]["precision"],
    "Recall": avg["best f1 under pa"]["recall"],

    "Event_F1": avg["event-based f1 under pa with mode squeeze"]["f1"],
    "Delay_F1": avg["best f1 under 5-delay pa"]["f1"],

    "AUPRC": avg["point-based auprc pa"],
}

print("\n=== ✅ Clean row (for comparison notebook) ===")
for k, v in row.items():
    print(f"{k}: {v}")

=== 📊 AVERAGE RESULTS (LSTM AE - SMAP) ===
best f1 under pa: {'f1': 0.5896740230815949, 'precision': 0.665706644723308, 'recall': 0.529228980029438, 'threshold': 2.837579905986786}
event-based f1 under pa with mode squeeze: {'f1': 0.167597765363128, 'precision': 0.13392857142857142, 'recall': 0.22388059701492538, 'threshold': 58.62755697965622}
best f1 under 5-delay pa: {'f1': 0.27178101503759355, 'precision': 0.20080028469998004, 'recall': 0.42038123966491614, 'threshold': 0.8557571768760681}
point-based auprc pa: 0.551319674876299

=== 📊 PER-SERIES RESULTS ===
Number of series: 1
Example entry:
('AllInOne', {'best f1 under pa': {'f1': 0.5896740230815949, 'precision': 0.665706644723308, 'recall': 0.529228980029438, 'threshold': 2.837579905986786}, 'event-based f1 under pa with mode squeeze': {'f1': 0.167597765363128, 'precision': 0.13392857142857142, 'recall': 0.22388059701492538, 'threshold': 58.62755697965622}, 'best f1 under 5-delay pa': {'f1': 0.27178101503759355, 'precision': 0.2